# Xenocomm core analysis on the reduced melanoma PDX dataset

This notebook loads the prepared dataset and trained model outputs from `01_train_model.ipynb`.

In [ ]:
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import xenocomm as xc

sns.set_theme(context="notebook", style="whitegrid")

def resolve_notebook_dir() -> Path:
    for name in ("__vsc_ipynb_file__", "__session__"):
        value = globals().get(name)
        if value:
            path = Path(value).expanduser()
            if path.suffix == ".ipynb":
                return path.resolve().parent
    if "__file__" in globals():
        return Path(__file__).resolve().parent
    raise RuntimeError("Could not determine the notebook path from this runner.")

NOTEBOOK_DIR = resolve_notebook_dir()
DATA_DIR = NOTEBOOK_DIR / "data/melanoma_pdx_10k"
OUTPUT_DIR = NOTEBOOK_DIR / "outputs"
MOUSE_H5AD = DATA_DIR / "adata_mouse.h5ad"
HUMAN_H5AD = DATA_DIR / "adata_human.h5ad"
MODEL_PATH = OUTPUT_DIR / "model.npz"
STAGED_PATH = OUTPUT_DIR / "model.staged.npz"

missing = [path for path in (MOUSE_H5AD, HUMAN_H5AD, MODEL_PATH) if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Expected prepared data and trained model outputs. Run the preparation script "
        f"and 01_train_model.ipynb first. Missing: {missing}"
    )

adata_mouse = ad.read_h5ad(MOUSE_H5AD)
adata_human = ad.read_h5ad(HUMAN_H5AD)

In [ ]:
if STAGED_PATH.exists():
    model, samples, var_dict = xc.load_staged_model(STAGED_PATH)
else:
    full_model = xc.XenocommModel.load(str(MODEL_PATH), adata_mouse, adata_human)
    samples = {key: np.asarray(value) for key, value in full_model.sample(100).items()}
    var_dict = {key: np.asarray(value) for key, value in full_model.get_parameters().items()}
    model = full_model

print(f"Mouse cells: {adata_mouse.n_obs:,}; human cells: {adata_human.n_obs:,}")
print(f"Ligands: {len(model.ligands):,}; receptors: {len(model.receptors):,}; targets: {len(model.targets):,}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sc.pl.umap(adata_mouse, color="cell_type", ax=axes[0], show=False, title="Mouse cell types")
sc.pl.umap(adata_human, color="cell_type", ax=axes[1], show=False, title="Human cell types")
plt.tight_layout()

In [ ]:
enrichment = xc.enrichment_df(model, samples, var_dict)
excluded = {"Farp2", "Fstl5", "Il1rapl1"}
enriched = (
    enrichment[
        (enrichment["prob_enriched"] >= 0.9)
        & (enrichment["human_fraction"] > 0.8)
        & (~enrichment["ligand"].isin(excluded))
    ]
    .sort_values("enrichment", ascending=False)
    .head(15)
)
enriched

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=enriched, x="enrichment", y="ligand", ax=ax, color="#4b9cbe")
ax.set_xlabel("Human counterfactual activation")
ax.set_ylabel("Ligand")
ax.set_title("Top enriched ligands")
plt.tight_layout()

In [ ]:
binding = xc.species_bias_df(model, samples, var_dict)
binding_top = binding[binding["ligand"].isin(enriched["ligand"])].copy()
binding_long = binding_top.melt(
    id_vars="ligand",
    value_vars=["mouse_binding", "human_binding"],
    var_name="species",
    value_name="binding",
)
binding_long["species"] = binding_long["species"].str.replace("_binding", "", regex=False).str.title()
binding_long["ligand"] = pd.Categorical(
    binding_long["ligand"], categories=enriched["ligand"].tolist(), ordered=True
)

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=binding_long, x="binding", y="ligand", hue="species", ax=ax)
ax.set_xlabel("Binding score")
ax.set_ylabel("Ligand")
ax.set_title("Species-decomposed ligand binding")
plt.tight_layout()

In [ ]:
receptor_marginal = xc.receptor_marginal_df(model, samples, var_dict)
receptor_top = receptor_marginal.sort_values("delta", ascending=False).head(12).copy()
receptor_long = receptor_top.melt(
    id_vars="receptor",
    value_vars=["mouse", "delta"],
    var_name="component",
    value_name="activation",
)
receptor_long["component"] = receptor_long["component"].map({"mouse": "Mouse baseline", "delta": "Human delta"})
receptor_long["receptor"] = pd.Categorical(
    receptor_long["receptor"], categories=receptor_top["receptor"].tolist(), ordered=True
)

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=receptor_long, x="activation", y="receptor", hue="component", ax=ax)
ax.set_xlabel("Activation")
ax.set_ylabel("Receptor")
ax.set_title("Top receptor marginal activation")
plt.tight_layout()

In [ ]:
top_ligand = str(enriched.iloc[0]["ligand"])
counterfactual = xc.receptor_counterfactual_df(
    model,
    samples,
    var_dict,
    remove_ligands=[top_ligand],
    label=top_ligand,
)

pivot = counterfactual.pivot(index="receptor", columns="component", values="activation").fillna(0)
pivot = pivot.loc[pivot.sum(axis=1).sort_values(ascending=False).head(12).index]

fig, ax = plt.subplots(figsize=(8, 4))
bottom = np.zeros(len(pivot))
for component in pivot.columns:
    values = pivot[component].to_numpy()
    ax.barh(pivot.index, values, left=bottom, label=component)
    bottom += values
ax.invert_yaxis()
ax.set_xlabel("Receptor activation")
ax.set_ylabel("Receptor")
ax.set_title(f"Counterfactual receptor activation: {top_ligand}")
ax.legend()
plt.tight_layout()

In [ ]:
targets = xc.targets_marginal_df(model, samples, var_dict, top_n=15)
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=targets, x="value", y="target", ax=ax, color="#7a9a01")
ax.set_xlabel("Weighted activation")
ax.set_ylabel("Target")
ax.set_title("Top downstream targets")
plt.tight_layout()
targets

In [ ]:
ligands = enriched["ligand"].head(8).tolist()
dot_data = xc.species_dotplot_data(
    model,
    samples,
    var_dict,
    adata_mouse,
    adata_human,
    ligands=ligands,
)
dot_df = pd.DataFrame(dot_data)

fig, ax = plt.subplots(figsize=(7, 3.5))
scatter = ax.scatter(
    dot_df["genes"],
    dot_df["species"],
    s=np.asarray(dot_df["size"]) * 500,
    c=dot_df["color"],
    cmap="viridis",
)
ax.set_xlabel("Ligand")
ax.set_ylabel("Species")
ax.set_title("Cross-species ligand expression")
plt.xticks(rotation=45, ha="right")
plt.colorbar(scatter, ax=ax, label="Mean expression")
plt.tight_layout()